# 03 - Kaggle parity + acceptance

Runs the live-vs-cached parity check, one real crop inference, and regenerates the acceptance reports on a GPU Kaggle runner. No retraining happens here.

Upload three files from the local checkout first (Kaggle dataset or manual upload):
- `data/embeddings/mll23_embeddings.pt` (~129 MB)
- `outputs/checkpoints/mlp.pt` (~1.6 MB)
- `outputs/reports/mll23_head_comparison.json` (a few KB)

Everything else is regenerated on the runner.

In [ ]:
import torch

assert torch.cuda.is_available(), "enable a GPU accelerator first"
print(torch.cuda.get_device_name(0), torch.__version__)

In [ ]:
!pip install -q timm torchvision Pillow numpy PyYAML pytest ruff mypy


## 1. Dataset + weights (idempotent, reuses the baseline flow)

In [ ]:
!PYTHONPATH=src python -m bloodfilm.cli assets download mll23 --report-output outputs/reports/mll23_download.json

In [ ]:
!PYTHONPATH=src python -m bloodfilm.cli assets verify mll23 --report-output outputs/reports/mll23_verify.json
!PYTHONPATH=src python -m bloodfilm.cli dataset build-manifest mll23 --output data/manifests/mll23_manifest.csv --invalid-output data/manifests/mll23_invalid.csv --report-output outputs/reports/mll23_manifest.json
!PYTHONPATH=src python -m bloodfilm.cli dataset split --manifest data/manifests/mll23_manifest.csv --output data/manifests/mll23_splits.csv --report-output outputs/reports/mll23_leakage.json --seed 42

In [ ]:
!mkdir -p models/backbones
!curl -L -o models/backbones/dinobloom-b.pth "https://huggingface.co/MarrLab/DinoBloom/resolve/main/pytorch_model_b.bin"
!echo "fdd8e51fd9831858c96cb241a694dd9d64cc28941ec2a923f1fa1e9b5c610a3f  models/backbones/dinobloom-b.pth" | sha256sum -c

In [ ]:
!PYTHONPATH=src python -m bloodfilm.cli smoke --config configs/base.yaml --image data/raw/MLL23/basophil/basophil/basophil_0001.TIF

## 2. Restore the training artifacts (upload these three files)

Copy the embeddings cache, the MLP checkpoint, and the head comparison into place. Replace `<your-dataset>` below with your Kaggle dataset slug. Fallback for the cache: re-extract embeddings on GPU with `embeddings extract` (frozen backbone, no retraining).

In [ ]:
!ls -lh /kaggle/input/
!mkdir -p data/embeddings outputs/checkpoints outputs/reports
!cp /kaggle/input/<your-dataset>/mll23_embeddings.pt data/embeddings/mll23_embeddings.pt
!cp /kaggle/input/<your-dataset>/mlp.pt outputs/checkpoints/mlp.pt
!cp /kaggle/input/<your-dataset>/mll23_head_comparison.json outputs/reports/mll23_head_comparison.json
!ls -lh data/embeddings/mll23_embeddings.pt outputs/checkpoints/mlp.pt outputs/reports/mll23_head_comparison.json

## 3. Evaluate the cached head (no retraining)

In [ ]:
!PYTHONPATH=src python -m bloodfilm.cli classifier evaluate-cache --checkpoint outputs/checkpoints/mlp.pt --embeddings data/embeddings/mll23_embeddings.pt --split test --output outputs/reports/mll23_test_evaluation.json --artifacts-dir outputs/reports

## 4. Package the bundle, then prove the live path

In [ ]:
!PYTHONPATH=src python -m bloodfilm.cli classifier package-bundle --checkpoint outputs/checkpoints/mlp.pt --comparison-report outputs/reports/mll23_head_comparison.json --bundle models/mll23-dinobloom-b-mlp-v0.1
!PYTHONPATH=src python -m bloodfilm.cli classifier inspect-bundle --bundle models/mll23-dinobloom-b-mlp-v0.1 --output outputs/reports/mll23_bundle_inspection.json

In [ ]:
!PYTHONPATH=src python -m bloodfilm.cli classifier parity-live --device cuda --output outputs/reports/parity_evidence.json
!mkdir -p outputs/predictions
!PYTHONPATH=src python -m bloodfilm.cli classifier classify-crop --bundle models/mll23-dinobloom-b-mlp-v0.1 --image data/raw/MLL23/basophil/basophil/basophil_0001.TIF --output outputs/predictions/parity_crop.json


## 5. Acceptance

In [ ]:
!PYTHONPATH=src python -m pytest -q tests/integration/test_parity.py 2>&1 | tail -n 3
!PYTHONPATH=src python -m ruff check src tests
!PYTHONPATH=src python -m mypy src 2>&1 | tail -n 2
!rm -f /tmp/verification.json
!PYTHONPATH=src python -m pytest -q && PYTHONPATH=src python -m ruff check src tests && PYTHONPATH=src python -m mypy src && python -c "import json; json.dump({'pytest': 'passed', 'ruff': 'passed', 'mypy': 'passed'}, open('/tmp/verification.json', 'w'))"
!PYTHONPATH=src python -m bloodfilm.cli classifier validation-reports --parity-evidence outputs/reports/parity_evidence.json --verification-report /tmp/verification.json


In [ ]:
import json
acceptance = json.loads(open("outputs/reports/mll23_m2_acceptance.json").read())
print("status:", acceptance["status"])
print("blockers:", acceptance["blockers"])
print("limitations:", acceptance.get("limitations"))
parity = json.loads(open("outputs/reports/crop_inference_parity.json").read())
print("parity:", parity["status"], "crops:", parity.get("crops_checked"))